In [0]:
from pyspark.sql import functions as F

customer_df = spark.table("customer_master")
rfm_df = spark.table("customer_rfm")
membership_df = spark.table("segment_membership")
recommendation_df = spark.table("customer_recommendation")
outcome_df = spark.table("action_outcome")
engagement_df = spark.table("engagement_event")
order_df = spark.table("order_master")
order_line_df = spark.table("order_line")

print("All analytics tables loaded successfully!")

In [0]:
#Customer segment summary
segment_summary_df = (
    membership_df
    .groupBy("segment_name")
    .agg(
        F.countDistinct("customer_id").alias("total_customers"),
        F.avg("rfm_score").alias("average_rfm_score")
    )
    .orderBy(F.desc("total_customers"))
)

display(segment_summary_df)

In [0]:
#save the table
segment_summary_df.write.format("delta").mode("overwrite").saveAsTable(
    "gold_segment_summary"
)

In [0]:
#Customer spending analysis
delivered_revenue_df = (
    order_df
    .filter(F.col("order_status") == "DELIVERED")
    .join(order_line_df, "order_id", "inner")
    .groupBy("customer_id")
    .agg(
        F.sum("line_total").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders")
    )
)

customer_spending_df = (
    rfm_df
    .join(delivered_revenue_df, "customer_id", "left")
    .fillna(0, subset=["total_revenue", "total_orders"])
)

display(customer_spending_df)

In [0]:
customer_spending_df.write.format("delta").mode("overwrite").saveAsTable(
    "gold_customer_spending"
)

In [0]:
#Recommendation performance
recommendation_performance_df = (
    outcome_df
    .groupBy("recommended_action")
    .agg(
        F.countDistinct("recommendation_id").alias("total_recommendations"),
        F.sum("purchase_count").alias("total_purchases"),
        F.sum(
            F.when(F.col("outcome_status") == "CONVERTED", 1).otherwise(0)
        ).alias("converted_recommendations"),
        F.sum("email_open_count").alias("total_email_opens"),
        F.sum("email_click_count").alias("total_email_clicks")
    )
    .withColumn(
        "conversion_rate",
        F.when(
            F.col("total_recommendations") > 0,
            F.round(
                F.col("converted_recommendations") * 100 /
                F.col("total_recommendations"), 2
            )
        ).otherwise(0)
    )
)

display(recommendation_performance_df)

In [0]:
recommendation_performance_df.write.format("delta").mode("overwrite").saveAsTable(
    "gold_recommendation_performance"
)

In [0]:
#Customer engagement analysis
engagement_summary_df = (
    engagement_df
    .groupBy("event_type")
    .agg(
        F.count("*").alias("total_events"),
        F.countDistinct("customer_id").alias("unique_customers")
    )
    .orderBy(F.desc("total_events"))
)

display(engagement_summary_df)

In [0]:
engagement_summary_df.write.format("delta").mode("overwrite").saveAsTable(
    "gold_engagement_summary"
)

In [0]:
#Check all Gold tables
gold_tables = [
    "gold_segment_summary",
    "gold_customer_spending",
    "gold_recommendation_performance",
    "gold_engagement_summary"
]

for table_name in gold_tables:
    print("Table:", table_name)
    print("Rows:", spark.table(table_name).count())

In [0]:
#Calculate business KPIs
total_customers = customer_df.count()

total_recommendations = recommendation_df.count()

converted_recommendations = outcome_df.filter(
    F.col("outcome_status") == "CONVERTED"
).count()

total_revenue = (
    delivered_revenue_df
    .agg(F.sum("total_revenue").alias("revenue"))
    .first()["revenue"]
)

conversion_rate = (
    converted_recommendations * 100 / total_recommendations
    if total_recommendations > 0 else 0
)

print("Total Customers:", total_customers)
print("Total Recommendations:", total_recommendations)
print("Converted Recommendations:", converted_recommendations)
print("Total Revenue:", total_revenue)
print("Conversion Rate:", round(conversion_rate, 2), "%")